In [3]:
import os
import time
os.environ['JAX_ENABLE_X64'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1' 

import jax
import jax.numpy as jnp
import math
import cupy as cp
import numba

import numpy as np
from temgym_core.components import Detector
from temgym_core.run import run_to_end
from temgym_core.source import circular_input_wave
from temgym_core.evaluate import (
    evaluate_gaussians_gpu_kernel_wrapper,
    evaluate_gaussians_for,
    evaluate_gaussians_jax_scan
)

from temgym_core.utils import FresnelPropagator
from numba import cuda
import numba

jax.config.update("jax_enable_x64", True)


In [ ]:
import os
import time
os.environ['JAX_ENABLE_X64'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1' 

import jax
import jax.numpy as jnp
import math
import cupy as cp
import numba

import numpy as np
from temgym_core.components import Detector
from temgym_core.run import run_to_end
from temgym_core.source import circular_input_wave
from temgym_core.evaluate import (
    evaluate_gaussians_gpu_kernel_wrapper,
    evaluate_gaussians_for,
    evaluate_gaussians_jax_scan
)

from temgym_core.utils import FresnelPropagator
from numba import cuda
import numba

jax.config.update("jax_enable_x64", True)


@cuda.jit(cache=True)
def _beam_field_cuda(x_det, y_det, r, dr, amp, pl, Qi_r, Qi_i, k, out_r, out_i):
    beam_idx, pix_idx = cuda.grid(2)

    if beam_idx >= r.shape[0] or pix_idx >= x_det.size:
        return

    dx = x_det[pix_idx] - r[beam_idx, 0]
    dy = y_det[pix_idx] - r[beam_idx, 1]

    linear = dx * dr[beam_idx, 0] + dy * dr[beam_idx, 1]

    quad_r = (
        Qi_r[beam_idx, 0, 0] * dx * dx
        + (Qi_r[beam_idx, 0, 1] + Qi_r[beam_idx, 1, 0]) * dx * dy
        + Qi_r[beam_idx, 1, 1] * dy * dy
    )
    quad_i = (
        Qi_i[beam_idx, 0, 0] * dx * dx
        + (Qi_i[beam_idx, 0, 1] + Qi_i[beam_idx, 1, 0]) * dx * dy
        + Qi_i[beam_idx, 1, 1] * dy * dy
    )

    s_real = pl[beam_idx] + linear + 0.5 * quad_r
    s_imag = 0.5 * quad_i

    phase = k[beam_idx] * s_real
    atten = math.exp(-k[beam_idx] * s_imag)
    c = math.cos(phase)
    s = math.sin(phase)

    ar = amp[beam_idx].real
    ai = amp[beam_idx].imag

    cuda.atomic.add(out_r, pix_idx, atten * (ar * c - ai * s))
    cuda.atomic.add(out_i, pix_idx, atten * (ar * s + ai * c))


def evaluate_gaussians_cuda_gpu_kernel_wrapper(gaussian_ray, grid, tpb_beam=4, tpb_pix=128):
    r, dr, amp, pl, Q_inv, k = _prepare_gaussian_params(gaussian_ray)

    x_det = np.ascontiguousarray(grid.coords[:, 0], dtype=np.float64)
    y_det = np.ascontiguousarray(grid.coords[:, 1], dtype=np.float64)

    n_beams = r.shape[0]
    n_pix = x_det.size

    Qi = np.asarray(Q_inv, dtype=np.complex128)

    d_x = cuda.to_device(x_det)
    d_y = cuda.to_device(y_det)
    d_r = cuda.to_device(np.asarray(r, dtype=np.float64))
    d_dr = cuda.to_device(np.asarray(dr, dtype=np.float64))
    d_amp = cuda.to_device(np.asarray(amp, dtype=np.complex128))
    d_pl = cuda.to_device(np.asarray(pl, dtype=np.float64))
    d_Qi_r = cuda.to_device(np.ascontiguousarray(Qi.real, dtype=np.float64))
    d_Qi_i = cuda.to_device(np.ascontiguousarray(Qi.imag, dtype=np.float64))
    d_k = cuda.to_device(np.asarray(k, dtype=np.float64))

    d_out_r = cp.zeros(n_pix, dtype=np.float64)
    d_out_i = cp.zeros(n_pix, dtype=np.float64)

    blocks = (math.ceil(n_beams / tpb_beam), math.ceil(n_pix / tpb_pix))
    _beam_field_cuda[blocks, (tpb_beam, tpb_pix)](
        d_x, d_y, d_r, d_dr, d_amp, d_pl, d_Qi_r, d_Qi_i, d_k, d_out_r, d_out_i
    )

    return (d_out_r.get() + 1j * d_out_i.get()).reshape(grid.shape)



def ensure_batch(x, sample_shape=(), dtype=None):
    """
    Ensure x has shape (B, *sample_shape), collapsing any existing leading dims into
    a batch dimension. If sample_shape == (), treat x as per-beam scalars and ensure
    shape (B,).

    Args:
        x: array-like
        sample_shape: tuple, the desired trailing shape
        dtype: optional dtype conversion

    Returns:
        JAX array with leading batch axis.
    """
    x = jnp.asarray(x, dtype=dtype)
    sample_shape = tuple(sample_shape)

    # Scalar case → produce (B,)
    if len(sample_shape) == 0:
        if x.ndim == 0:
            return x[None]           # scalar → (1,)
        if x.ndim == 1:
            return x                 # already (B,)
        return x.reshape((-1,))      # collapse all dims → (B,)

    # Non-scalar case
    s = len(sample_shape)

    # If trailing dims already match sample_shape → collapse leading dims to batch
    if x.ndim >= s and tuple(x.shape[-s:]) == sample_shape:
        return x.reshape((-1,) + sample_shape)

    # Exact match to sample_shape → add leading batch axis
    if tuple(x.shape) == sample_shape:
        return x[None, ...]

    # Fallback: add leading axis
    return x[None, ...]


def _prepare_gaussian_params(gaussian_ray):
    r = ensure_batch(gaussian_ray.r_xy, (2,), jnp.float64)
    dr = ensure_batch(gaussian_ray.d_xy, (2,), jnp.float64)
    amplitude = ensure_batch(gaussian_ray.amplitude, (), jnp.complex128)
    pathlength = ensure_batch(gaussian_ray.pathlength, (), jnp.float64)
    Q_inv = ensure_batch(gaussian_ray.Q_inv,  (2, 2), jnp.complex128)
    k = ensure_batch(gaussian_ray.k, (), jnp.float64)

    return r, dr, amplitude, pathlength, Q_inv, k

# Block dims are compile-time constants so cuda.shared.array can use them
_OPT_TPB_BEAM = 32
_OPT_TPB_PIX  = 128


@cuda.jit(cache=True, fastmath=True, lineinfo=True)
def _beam_field_cuda_opt(x_det, y_det, rays, out_r, out_i):
    pix_idx, beam_block_idx = cuda.grid(2)
    # loc_beam_block = cuda.threadIdx.y  -> always zero
    # loc_pix  = cuda.threadIdx.x  # fast idx

    # for accuracy, keep these f64
    acc_r = np.float64(0.0)
    acc_i = np.float64(0.0)

    for loc_beam in range(_OPT_TPB_BEAM):
        beam_idx  = beam_block_idx * _OPT_TPB_BEAM + loc_beam
        if beam_idx < rays['r'].shape[0] and pix_idx < x_det.size:
            ray = rays[beam_idx]
            # x_det, y_det -> load coalescing on fast index
            dx = x_det[pix_idx] - ray['r'][0]
            dy = y_det[pix_idx] - ray['r'][1]

            linear = dx * ray['dr'][0] + dy * ray['dr'][1]

            quad_r = (
                ray['Qi_r'][0] * dx * dx
                + ray['Qi_r'][1] * dx * dy
                + ray['Qi_r'][2] * dy * dy
            )
            quad_i = (
                ray['Qi_i'][0] * dx * dx
                + ray['Qi_i'][1] * dx * dy
                + ray['Qi_i'][2] * dy * dy
            )

            s_real = ray['pl'] + linear + 0.5 * quad_r
            s_imag = 0.5 * quad_i

            atten = math.exp(-ray['k'] * s_imag)
            phase = ray['k'] * s_real
            c = math.cos(phase)
            s = math.sin(phase)

            ar = ray['amp'].real
            ai = ray['amp'].imag

            acc_r += atten * (ar * c - ai * s)
            acc_i += atten * (ar * s + ai * c)

    cuda.syncthreads()

    # One thread per pixel reduces the beam dimension → one global atomic per block per pixel
    if pix_idx < x_det.size:
        cuda.atomic.add(out_r, pix_idx, acc_r)
        cuda.atomic.add(out_i, pix_idx, acc_i)


rays_type = np.dtype([
    ('Qi_r', np.float32, (3,)),  # packed: (0,0), (0,1)+(1,0), (1,1)
    ('Qi_i', np.float32, (3,)),
    ('amp', np.complex64),
    ('pl', np.float64),
    ('r', np.float32, (2,)),
    ('dr', np.float32, (2,)),
    ('k', np.float32),
    ('PADDING', np.float32),
])

assert rays_type.itemsize == 64

def evaluate_gaussians_cuda_gpu_kernel_wrapper_opt(gaussian_ray, grid):
    r, dr, amp, pl, Q_inv, k = _prepare_gaussian_params(gaussian_ray)

    x_det = np.ascontiguousarray(grid.coords[:, 0], dtype=np.float32)
    y_det = np.ascontiguousarray(grid.coords[:, 1], dtype=np.float32)

    n_beams = r.shape[0]
    n_pix   = x_det.size

    Qi = np.asarray(Q_inv, dtype=np.complex64)

    d_x    = cuda.to_device(x_det)
    d_y    = cuda.to_device(y_det)

    rays = np.zeros(dtype=rays_type, shape=(n_beams,))
    rays['Qi_r'] = np.array([Qi.real[:, 0, 0], Qi.real[:, 0, 1] + Qi.real[:, 1, 0], Qi.real[:, 1, 1]]).T
    rays['Qi_i'] = np.array([Qi.imag[:, 0, 0], Qi.imag[:, 0, 1] + Qi.imag[:, 1, 0], Qi.imag[:, 1, 1]]).T
    rays['r'] = r
    rays['dr'] = dr
    rays['amp'] = amp
    rays['pl'] = pl
    rays['k'] = k

    d_rays = cuda.to_device(rays)

    d_out_r = cp.zeros(n_pix, dtype=np.float64)
    d_out_i = cp.zeros(n_pix, dtype=np.float64)

    blocks = (math.ceil(n_pix / _OPT_TPB_PIX), math.ceil(n_beams / _OPT_TPB_BEAM),)
    t0 = time.time()
    _beam_field_cuda_opt[blocks, (_OPT_TPB_PIX, 1)](
        d_x, d_y, d_rays, d_out_r, d_out_i
    )
    res = (d_out_r.get() + 1j * d_out_i.get()).reshape(grid.shape)
    t1 = time.time()
    print(f"kernel only: {t1-t0}")
    return res



if __name__ == "__main__":
    W = 10e-6  # Width of the simulation cell in nm

    Nx = Ny = 256
    center_idx = Nx // 2
    dx = W/Nx
    dy = W/Ny
    grid = Detector(z=1e-1, pixel_size=(dx, dy), shape=(Nx, Ny))
    coords = grid.coords
    X, Y = coords[:,0].reshape(grid.shape), coords[:,1].reshape(grid.shape)
    x, y = X[0,:], Y[:,0]
    extent = (x[0], x[-1], y[0], y[-1])

    voltage = 100e3  # in volts
    components = (grid,)

    rays_in = circular_input_wave(
        voltage=voltage,
        aperture_radius=W/4,
        waist=2e-8
    )
    print(f"number of rays: {rays_in.x.size}")
    print(f"rays x Nx x Ny: {rays_in.x.size * Nx * Ny}")
    rays_in = rays_in.to_vector()

    r, dr, amp, pathlength, Q_inv, k = _prepare_gaussian_params(rays_in)
    r2 = grid.coords
    n = r.shape[0]
    total_field = jnp.zeros((r2.shape[0],), dtype=jnp.complex128)
    evaluate_gaussians_cuda_gpu_kernel_wrapper_opt(rays_in, grid)

    t0 = time.time()
    res_opt = evaluate_gaussians_cuda_gpu_kernel_wrapper_opt(rays_in, grid)
    t1 = time.time()
    print(f"time w/ transfers: {t1-t0}")

    res = evaluate_gaussians_cuda_gpu_kernel_wrapper(rays_in, grid)

    from numpy.testing import assert_allclose

    print(f"max={res.max()}, min={res.min()}")

    assert_allclose(res, res_opt, atol=5e-6)

number of rays: 196350
rays x Nx x Ny: 12867993600
kernel only: 0.13059496879577637
kernel only: 0.1291508674621582
time w/ transfers: 0.16417789459228516
max=(1.0019523552537615+0j), min=0j
